# Loss Functions for Machine Learning — Theory Notebook

> _"To define is to choose a loss function. Every other choice follows."_

Interactive derivations and visualisations covering regression losses, classification losses,
probabilistic and contrastive losses, loss landscape geometry, and modern AI objectives.

**Run cells in order.** Each section builds on the previous.

In [ ]:
import numpy as np
import numpy.linalg as la
from scipy import stats
from scipy.special import expit as sigmoid
from scipy.linalg import sqrtm

try:
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    try:
        import seaborn as sns
        sns.set_theme(style='whitegrid', palette='colorblind')
        HAS_SNS = True
    except ImportError:
        plt.style.use('seaborn-v0_8-whitegrid')
        HAS_SNS = False
    mpl.rcParams.update({
        'figure.figsize': (10, 6), 'figure.dpi': 100,
        'font.size': 12, 'axes.titlesize': 14, 'axes.labelsize': 12,
        'lines.linewidth': 2.0, 'axes.spines.top': False, 'axes.spines.right': False,
    })
    COLORS = {'primary': '#0077BB', 'secondary': '#EE7733', 'tertiary': '#009988',
              'error': '#CC3311', 'neutral': '#555555', 'highlight': '#EE3377'}
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    COLORS = {}

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def check_close(name, got, expected, tol=1e-6):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print(f'  expected: {expected}')
        print(f'  got:      {got}')
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

print('Setup complete. NumPy:', np.__version__)

---

## 1. What a Loss Function Does: Visualising the Choice

Three models trained on the same bimodal data but with different losses make very different predictions.
MSE predicts the mean (often a value that never occurs); MAE predicts the median; NLL models the full distribution.

In [ ]:
# === 1.1 Bimodal Data: MSE vs MAE vs Median ===

np.random.seed(42)
# Bimodal data: 50% chance of 0, 50% chance of 10
N = 1000
y_data = np.concatenate([np.zeros(N//2), np.ones(N//2) * 10])
np.random.shuffle(y_data)

# Optimal constant predictors
def mse_optimal(y): return np.mean(y)
def mae_optimal(y): return np.median(y)
def mode_optimal(y): return stats.mode(y, keepdims=True).mode[0]

pred_mse  = mse_optimal(y_data)
pred_mae  = mae_optimal(y_data)

# Losses at each predictor
def mse_loss(pred, y): return np.mean((y - pred)**2)
def mae_loss(pred, y): return np.mean(np.abs(y - pred))

# Scan over prediction values
preds = np.linspace(-1, 11, 300)
mse_vals = [mse_loss(p, y_data) for p in preds]
mae_vals = [mae_loss(p, y_data) for p in preds]

print(f'Bimodal data: {N//2} zeros + {N//2} tens')
print(f'MSE optimal predictor: {pred_mse:.2f}  (mean — never occurs!)')
print(f'MAE optimal predictor: {pred_mae:.2f}  (median)')

check_close('MSE minimiser is mean', pred_mse, 5.0, tol=0.1)
check_true('MAE minimiser is in {0, 10}', pred_mae in [0.0, 5.0, 10.0])

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    ax1.hist(y_data, bins=20, density=True, alpha=0.6, color=COLORS['primary'])
    ax1.axvline(pred_mse, color=COLORS['error'], linewidth=2.5, label=f'MSE opt = {pred_mse:.1f}')
    ax1.axvline(pred_mae, color=COLORS['secondary'], linewidth=2.5, label=f'MAE opt = {pred_mae:.1f}')
    ax1.set_title('Bimodal Data Distribution'); ax1.legend(); ax1.set_xlabel('y')
    ax2.plot(preds, mse_vals, color=COLORS['error'], label='MSE loss')
    ax2.plot(preds, mae_vals, color=COLORS['secondary'], label='MAE loss')
    ax2.axvline(pred_mse, color=COLORS['error'], linestyle='--', alpha=0.6)
    ax2.axvline(pred_mae, color=COLORS['secondary'], linestyle='--', alpha=0.6)
    ax2.set_title('Loss vs Predictor Value'); ax2.legend(); ax2.set_xlabel('Predicted value')
    plt.suptitle('Loss Choice Determines Optimal Prediction', fontsize=14)
    plt.tight_layout(); plt.show()

---

## 2. Bayes Risk and Optimal Predictors

The **Bayes optimal predictor** minimises the expected loss — its form depends entirely on the loss:

| Loss | Optimal predictor $f^*$ |
|---|---|
| MSE | $\mathbb{E}[y \mid \mathbf{x}]$ — conditional mean |
| MAE | $\operatorname{median}(y \mid \mathbf{x})$ — conditional median |
| Quantile $\tau$ | $\tau$-quantile of $P(y \mid \mathbf{x})$ |
| Cross-entropy | $P(y \mid \mathbf{x})$ — true conditional distribution |

In [ ]:
# === 2.1 Bayes Risk Verification ===

# Conditional distribution: y|x ~ 0.3*N(0,1) + 0.7*N(5,1)
def conditional_sample(x, n=10000):
    """Sample y|x from a mixture Gaussian (x ignored for toy demo)."""
    mix = np.random.rand(n) < 0.3
    return np.where(mix, np.random.randn(n), 5 + np.random.randn(n))

y_samples = conditional_sample(None)

# Scan all predictors
preds_scan = np.linspace(-2, 8, 500)
mse_risks  = [np.mean((y_samples - p)**2) for p in preds_scan]
mae_risks  = [np.mean(np.abs(y_samples - p)) for p in preds_scan]
q90_risks  = [np.mean(np.where(y_samples >= p, 0.9*(y_samples-p), 0.1*(p-y_samples)))
              for p in preds_scan]

opt_mse = preds_scan[np.argmin(mse_risks)]
opt_mae = preds_scan[np.argmin(mae_risks)]
opt_q90 = preds_scan[np.argmin(q90_risks)]

# Analytic values for this mixture
true_mean   = 0.3*0 + 0.7*5  # = 3.5
true_median = np.median(y_samples)
true_q90    = np.percentile(y_samples, 90)

print('Optimal predictors for 0.3*N(0,1) + 0.7*N(5,1):')
print(f'  MSE (expect mean={true_mean:.2f}):   scan={opt_mse:.3f}')
print(f'  MAE (expect median~{true_median:.2f}): scan={opt_mae:.3f}')
print(f'  Q90 (expect q90~{true_q90:.2f}):   scan={opt_q90:.3f}')

check_close('MSE minimiser ≈ conditional mean', opt_mse, true_mean, tol=0.1)
check_close('MAE minimiser ≈ conditional median', opt_mae, true_median, tol=0.1)
check_close('Q90 minimiser ≈ 90th percentile', opt_q90, true_q90, tol=0.2)

if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, risks, opt, label, color in zip(
            axes,
            [mse_risks, mae_risks, q90_risks],
            [opt_mse, opt_mae, opt_q90],
            ['MSE (mean)', 'MAE (median)', 'Quantile 0.9'],
            [COLORS['error'], COLORS['secondary'], COLORS['tertiary']]):
        ax.plot(preds_scan, risks, color=color, linewidth=2)
        ax.axvline(opt, color='black', linestyle='--', linewidth=1.5)
        ax.set_title(f'{label}\nOptimum = {opt:.2f}')
        ax.set_xlabel('Prediction')
    plt.suptitle('Different Losses → Different Optimal Predictors', fontsize=13)
    plt.tight_layout(); plt.show()

---

## 3. Regression Losses: Gradients and Robustness

The key geometric difference between regression losses is how they grow with the residual $r = \hat{y} - y$:
- **MSE**: quadratic growth — large residuals dominate training
- **MAE**: linear growth — all residuals contribute equally
- **Huber**: quadratic near zero, linear for outliers — best of both
- **Log-cosh**: smooth version of Huber

In [ ]:
# === 3.1 Regression Loss Zoo ===

def mse(r):     return r**2
def mae(r):     return np.abs(r)
def huber(r, delta=1.0):
    return np.where(np.abs(r) <= delta,
                    0.5*r**2,
                    delta*np.abs(r) - 0.5*delta**2)
def logcosh(r): return np.log(np.cosh(r))
def quantile(r, tau=0.9):
    return np.where(r >= 0, tau*r, (tau-1)*r)

def grad_mse(r):  return 2*r
def grad_mae(r):  return np.sign(r)
def grad_huber(r, delta=1.0):
    return np.where(np.abs(r) <= delta, r, delta*np.sign(r))
def grad_logcosh(r): return np.tanh(r)

r = np.linspace(-4, 4, 400)

print('Loss values at r = 2 (moderate outlier):')
r_test = 2.0
for name, fn in [('MSE', mse), ('MAE', mae), ('Huber(1)', huber), ('Log-Cosh', logcosh)]:
    val = fn(r_test) if name != 'Huber(1)' else huber(r_test, 1.0)
    print(f'  {name:<12}: {val:.4f}')

print('\nGradient magnitudes at r = 2:')
for name, fn in [('MSE', grad_mse), ('MAE', grad_mae),
                  ('Huber(1)', lambda r: grad_huber(r, 1.0)), ('Log-Cosh', grad_logcosh)]:
    val = fn(r_test)
    print(f'  {name:<12}: {val:.4f}')

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    for name, fn, col in [
            ('MSE', mse, COLORS['error']),
            ('MAE', mae, COLORS['secondary']),
            ('Huber(δ=1)', lambda r: huber(r, 1.0), COLORS['primary']),
            ('Log-Cosh', logcosh, COLORS['tertiary'])]:
        ax1.plot(r, fn(r), label=name, linewidth=2)
        ax2.plot(r, grad_mse(r) if name=='MSE' else
                    grad_mae(r) if name=='MAE' else
                    grad_huber(r,1.0) if 'Huber' in name else
                    grad_logcosh(r), label=name, linewidth=2)
    ax1.set_title('Loss Value vs Residual'); ax1.set_xlabel('r = ŷ - y')
    ax1.set_ylim(-0.5, 8); ax1.legend()
    ax2.set_title('Gradient vs Residual'); ax2.set_xlabel('r = ŷ - y')
    ax2.axhline(0, color='gray', linewidth=0.5); ax2.legend()
    plt.suptitle('Regression Losses: Loss and Gradient Comparison', fontsize=13)
    plt.tight_layout(); plt.show()

In [ ]:
# === 3.2 Outlier Robustness Demo ===

np.random.seed(42)
n = 50
x = np.linspace(0, 5, n)
y_clean = 2*x + 1 + np.random.randn(n)*0.5

# Add 3 extreme outliers
y_corrupted = y_clean.copy()
outlier_idx = [10, 25, 40]
y_corrupted[outlier_idx] = [20, -5, 25]

from numpy.polynomial import polynomial as P

# Fit using gradient descent for MSE and MAE
def fit_linear_gd(x, y, loss_type='mse', n_iter=5000, lr=1e-3):
    w, b = 0.0, 0.0
    for _ in range(n_iter):
        r = w*x + b - y
        if loss_type == 'mse':
            gw = 2*np.mean(r*x)
            gb = 2*np.mean(r)
        elif loss_type == 'mae':
            gw = np.mean(np.sign(r)*x)
            gb = np.mean(np.sign(r))
        elif loss_type == 'huber':
            g = np.where(np.abs(r)<=1, r, np.sign(r))
            gw = np.mean(g*x)
            gb = np.mean(g)
        w -= lr*gw; b -= lr*gb
    return w, b

w_mse, b_mse = fit_linear_gd(x, y_corrupted, 'mse')
w_mae, b_mae = fit_linear_gd(x, y_corrupted, 'mae', lr=5e-4)
w_hub, b_hub = fit_linear_gd(x, y_corrupted, 'huber', lr=5e-4)

print('Fitted parameters (true: w=2, b=1):')
print(f'  MSE:   w={w_mse:.3f}, b={b_mse:.3f}')
print(f'  MAE:   w={w_mae:.3f}, b={b_mae:.3f}')
print(f'  Huber: w={w_hub:.3f}, b={b_hub:.3f}')

check_true('Huber closer to true w=2 than MSE (with outliers)',
           abs(w_hub - 2) < abs(w_mse - 2))

if HAS_MPL:
    plt.figure(figsize=(10, 5))
    plt.scatter(x, y_corrupted, alpha=0.5, label='Data', color=COLORS['neutral'])
    plt.scatter(x[outlier_idx], y_corrupted[outlier_idx],
                color=COLORS['error'], s=100, zorder=5, label='Outliers')
    xx = np.array([0, 5])
    plt.plot(xx, w_mse*xx+b_mse, color=COLORS['error'], label=f'MSE fit: w={w_mse:.2f}')
    plt.plot(xx, w_mae*xx+b_mae, color=COLORS['secondary'], label=f'MAE fit: w={w_mae:.2f}')
    plt.plot(xx, w_hub*xx+b_hub, color=COLORS['primary'], label=f'Huber fit: w={w_hub:.2f}')
    plt.plot(xx, 2*xx+1, 'k--', linewidth=1.5, label='True: w=2')
    plt.title('Outlier Robustness: MSE vs MAE vs Huber')
    plt.legend(); plt.xlabel('x'); plt.ylabel('y')
    plt.tight_layout(); plt.show()

---

## 4. NLL as Unifying Framework: MLE Derivations

Every standard loss is the NLL of a specific noise model.
We verify this numerically for MSE (Gaussian) and BCE (Bernoulli).

In [ ]:
# === 4.1 MSE = Gaussian NLL (numerical verification) ===

np.random.seed(42)
n = 1000
sigma = 1.5
x_data = np.random.randn(n)
y_data = 2*x_data + 1 + sigma*np.random.randn(n)

# Gaussian log-likelihood (up to constant)
def gaussian_nll(w, b, x, y, sigma):
    residuals = y - (w*x + b)
    return 0.5*np.mean(residuals**2 / sigma**2) + 0.5*np.log(2*np.pi*sigma**2)

# MSE
def mse_loss(w, b, x, y):
    return np.mean((y - (w*x + b))**2)

# Both should be minimised at the same (w, b)
# Grid search
ws = np.linspace(1.5, 2.5, 50)
bs = np.linspace(0.5, 1.5, 50)
W, B = np.meshgrid(ws, bs)

nll_vals = np.array([[gaussian_nll(w, b, x_data, y_data, sigma) for w in ws] for b in bs])
mse_vals = np.array([[mse_loss(w, b, x_data, y_data) for w in ws] for b in bs])

nll_opt = (bs[nll_vals.argmin()//len(ws)], ws[nll_vals.argmin()%len(ws)])
mse_opt = (bs[mse_vals.argmin()//len(ws)], ws[mse_vals.argmin()%len(ws)])

print('Optimal parameters:')
print(f'  Gaussian NLL:  w={nll_opt[1]:.3f}, b={nll_opt[0]:.3f}')
print(f'  MSE:           w={mse_opt[1]:.3f}, b={mse_opt[0]:.3f}')
print(f'  True values:   w=2.000, b=1.000')
check_close('NLL and MSE minimised at same parameters', nll_opt, mse_opt, tol=0.1)

In [ ]:
# === 4.2 BCE = Bernoulli NLL (numerical verification) ===

np.random.seed(42)
n = 2000

# True model: P(y=1|x) = sigmoid(3x - 1)
x_cls = np.random.randn(n) * 2
p_true = sigmoid(3*x_cls - 1)
y_cls = (np.random.rand(n) < p_true).astype(float)

def bce_from_logit(w, b, x, y):
    z = w*x + b
    # Numerically stable: max(z,0) - y*z + log(1+exp(-|z|))
    return np.mean(np.maximum(z, 0) - y*z + np.log1p(np.exp(-np.abs(z))))

def bernoulli_nll(w, b, x, y):
    p = sigmoid(w*x + b).clip(1e-7, 1-1e-7)
    return -np.mean(y*np.log(p) + (1-y)*np.log(1-p))

# Verify they are identical
w_test, b_test = 2.5, -0.8
bce_val = bce_from_logit(w_test, b_test, x_cls, y_cls)
nll_val = bernoulli_nll(w_test, b_test, x_cls, y_cls)

print(f'BCE (stable):     {bce_val:.8f}')
print(f'Bernoulli NLL:    {nll_val:.8f}')
check_close('BCE == Bernoulli NLL', bce_val, nll_val)

# Find MLE via grid search
ws_cls = np.linspace(2, 4, 40)
bs_cls = np.linspace(-2, 0, 40)
nll_grid = np.array([[bernoulli_nll(w,b,x_cls,y_cls) for w in ws_cls] for b in bs_cls])
opt_idx = np.unravel_index(nll_grid.argmin(), nll_grid.shape)
w_mle, b_mle = ws_cls[opt_idx[1]], bs_cls[opt_idx[0]]
print(f'\nMLE estimate: w={w_mle:.2f}, b={b_mle:.2f}  (true: w=3, b=-1)')
check_true('MLE w ≈ 3', abs(w_mle - 3) < 0.5)
check_true('MLE b ≈ -1', abs(b_mle - (-1)) < 0.5)

---

## 5. Cross-Entropy: The Canonical Classification Loss

**Key equation**: For logits $\mathbf{z}$ with $\hat{\mathbf{p}} = \operatorname{softmax}(\mathbf{z})$:
$$\frac{\partial \ell_{\text{CE}}}{\partial z_k} = \hat{p}_k - y_k$$

The gradient is the prediction residual — clean and numerically stable.

In [ ]:
# === 5.1 Softmax-CE Gradient Verification ===

def softmax(z):
    z = z - z.max()  # numerically stable
    e = np.exp(z)
    return e / e.sum()

def cross_entropy(z, y_onehot):
    p = softmax(z)
    return -np.sum(y_onehot * np.log(p.clip(1e-10)))

# Verify gradient analytically
K = 5
z = np.array([2.0, 1.0, 0.5, -0.5, -1.0])
y_star = 1  # true class
y_onehot = np.eye(K)[y_star]
p_hat = softmax(z)

# Analytic gradient
grad_analytic = p_hat - y_onehot

# Numerical gradient (finite differences)
eps = 1e-5
grad_numerical = np.zeros(K)
for k in range(K):
    z_plus = z.copy(); z_plus[k] += eps
    z_minus = z.copy(); z_minus[k] -= eps
    grad_numerical[k] = (cross_entropy(z_plus, y_onehot) - cross_entropy(z_minus, y_onehot)) / (2*eps)

print('Softmax-CE gradient comparison:')
print(f'  Analytic (p_hat - y): {grad_analytic}')
print(f'  Numerical (finite diff): {grad_numerical}')
check_close('Analytic == numerical gradient', grad_analytic, grad_numerical, tol=1e-5)

# Verify at perfect prediction: gradient = 0
z_perfect = np.array([100.0, -100.0, -100.0, -100.0, -100.0])
p_perfect = softmax(z_perfect)
y_class0 = np.eye(K)[0]
grad_perfect = p_perfect - y_class0
print(f'\nGradient when perfectly correct: {grad_perfect}')
check_close('Gradient ≈ 0 when perfectly calibrated', grad_perfect, np.zeros(K), tol=1e-5)

In [ ]:
# === 5.2 Numerical Stability: log_softmax vs softmax+log ===

def unstable_log_softmax(z):
    """WRONG: softmax first, then log."""
    return np.log(np.exp(z) / np.sum(np.exp(z)))

def stable_log_softmax(z):
    """CORRECT: subtract max before exponentiating."""
    m = z.max()
    return z - m - np.log(np.sum(np.exp(z - m)))

# Test with large logits
print('Log-softmax stability test:')
for z_test in [np.array([1.0, 2.0, 3.0]),
               np.array([100.0, 200.0, 300.0]),
               np.array([-100.0, -200.0, -300.0])]:
    stable = stable_log_softmax(z_test)
    try:
        unstable = unstable_log_softmax(z_test)
        unstable_str = str(np.round(unstable, 4))
    except:
        unstable_str = 'NaN/Inf'
    print(f'  z = {z_test}:')
    print(f'    stable:   {np.round(stable, 4)}')
    print(f'    unstable: {unstable_str}')

# Verify stable version gives correct probabilities
z = np.array([1.0, 2.0, 3.0])
log_p = stable_log_softmax(z)
check_close('log-softmax probabilities sum to 1', np.exp(log_p).sum(), 1.0)
check_close('stable == scipy softmax', np.exp(log_p),
            np.exp(z - z.max()) / np.sum(np.exp(z - z.max())))

---

## 6. Classification Losses: Hinge, Focal, Label Smoothing

We visualise the hinge loss, focal loss, and label smoothing effects on gradients and calibration.

In [ ]:
# === 6.1 Hinge vs Cross-Entropy: Margin Analysis ===

import numpy as np
from scipy.special import expit as sigmoid

try:
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    HAS_MPL = True
    COLORS = {'primary': '#0077BB', 'secondary': '#EE7733', 'tertiary': '#009988',
              'error': '#CC3311', 'neutral': '#555555', 'highlight': '#EE3377'}
except ImportError:
    HAS_MPL = False

# Margin z = y * f(x) where y in {-1,+1}
z = np.linspace(-3, 3, 400)

def hinge(z):        return np.maximum(0, 1 - z)
def sq_hinge(z):     return np.maximum(0, 1 - z)**2
def log_loss(z):     return np.log1p(np.exp(-z))
def exp_loss(z):     return np.exp(-z)
def zero_one(z):     return (z < 0).astype(float)

print('Loss values at margin z = 1 (on boundary):')
for name, fn in [('0-1', zero_one), ('Hinge', hinge), ('Sq-Hinge', sq_hinge),
                  ('Log-loss', log_loss), ('Exp', exp_loss)]:
    print(f'  {name:<12}: {fn(1.0):.4f}')

print('\nLoss values at margin z = -1 (wrong side):')
for name, fn in [('0-1', zero_one), ('Hinge', hinge), ('Sq-Hinge', sq_hinge),
                  ('Log-loss', log_loss), ('Exp', exp_loss)]:
    print(f'  {name:<12}: {fn(-1.0):.4f}')

if HAS_MPL:
    plt.figure(figsize=(10, 5))
    plt.plot(z, zero_one(z), 'k--', linewidth=1.5, label='0-1 loss (non-convex)')
    plt.plot(z, hinge(z), color=COLORS['error'], label='Hinge')
    plt.plot(z, log_loss(z), color=COLORS['primary'], label='Log-loss (BCE)')
    plt.plot(z, exp_loss(z).clip(0, 5), color=COLORS['secondary'], label='Exponential')
    plt.axvline(1, color='gray', linewidth=0.5, linestyle=':')
    plt.axvline(0, color='gray', linewidth=0.5, linestyle=':')
    plt.xlabel('Margin z = y·f(x)'); plt.ylabel('Loss')
    plt.title('Surrogate Losses as Upper Bounds on 0-1 Loss')
    plt.ylim(-0.1, 4); plt.legend(); plt.tight_layout(); plt.show()

# Verify: all surrogates >= 0-1 loss
z_test = np.linspace(-5, 5, 1000)
ok = np.all(hinge(z_test) >= zero_one(z_test))
print(f"\n{'PASS' if ok else 'FAIL'} — Hinge >= 0-1 loss everywhere")
ok2 = np.all(log_loss(z_test) >= zero_one(z_test))
print(f"{'PASS' if ok2 else 'FAIL'} — Log-loss >= 0-1 loss everywhere")

In [ ]:
# === 6.2 Focal Loss: Down-weighting Easy Examples ===

def focal_loss(p, gamma=2.0):
    """Binary focal loss for positive class. p is P(y=1)."""
    return -(1 - p)**gamma * np.log(p.clip(1e-7))

def cross_entropy_loss(p):
    return -np.log(p.clip(1e-7))

p_range = np.linspace(0.01, 0.99, 300)

print('Focal loss relative weight vs CE:')
print(f'  p=0.95 (easy): FL/CE = {focal_loss(np.array([0.95]))[0]/cross_entropy_loss(np.array([0.95]))[0]:.4f}')
print(f'  p=0.50 (hard): FL/CE = {focal_loss(np.array([0.50]))[0]/cross_entropy_loss(np.array([0.50]))[0]:.4f}')
print(f'  p=0.05 (very hard): FL/CE = {focal_loss(np.array([0.05]))[0]/cross_entropy_loss(np.array([0.05]))[0]:.4f}')

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    ax1.plot(p_range, cross_entropy_loss(p_range), color=COLORS['neutral'],
             linewidth=2, label='CE (γ=0)')
    for gamma, col in [(0.5, COLORS['tertiary']), (1.0, COLORS['secondary']),
                       (2.0, COLORS['primary']), (5.0, COLORS['error'])]:
        ax1.plot(p_range, focal_loss(p_range, gamma), color=col, label=f'Focal γ={gamma}')
    ax1.set_xlabel('Predicted P(y=1)'); ax1.set_ylabel('Loss')
    ax1.set_title('Focal Loss: Effect of γ'); ax1.legend(); ax1.set_ylim(0, 5)

    # Relative weight: how much is a correct-confident prediction down-weighted?
    ratios_gamma2 = focal_loss(p_range, 2.0) / cross_entropy_loss(p_range)
    ax2.plot(p_range, ratios_gamma2, color=COLORS['primary'], linewidth=2)
    ax2.set_xlabel('Predicted P(y=1)'); ax2.set_ylabel('Focal / CE ratio')
    ax2.set_title('Down-weighting Factor (γ=2): Easy→0, Hard→1')
    ax2.axhline(1.0, color='gray', linestyle='--', linewidth=0.8)
    plt.suptitle('Focal Loss: Mechanism of Hard-Example Mining', fontsize=13)
    plt.tight_layout(); plt.show()

In [ ]:
# === 6.3 Label Smoothing: Effect on Confidence ===

K = 10  # number of classes

def label_smooth_ce(logits, y_star, epsilon=0.1):
    """CE loss with label smoothing."""
    z_shifted = logits - logits.max()
    log_p = z_shifted - np.log(np.sum(np.exp(z_shifted)))
    # Smoothed target
    y = np.full(K, epsilon / K)
    y[y_star] += 1 - epsilon
    return -np.sum(y * log_p)

def hard_ce(logits, y_star):
    """Standard CE loss."""
    z_shifted = logits - logits.max()
    log_p = z_shifted - np.log(np.sum(np.exp(z_shifted)))
    return -log_p[y_star]

# Simulate training: optimal logit gap analysis
# With label smoothing eps=0.1, K=10:
# Optimal logit gap = log((1-eps)/(eps/K)) - log(eps/(K*(K-1)))
# Approximately: target logit >> other logits but not infinitely

gaps = np.linspace(0, 10, 200)
base_logit = np.zeros(K)

hard_losses = []
smooth_losses = []
for gap in gaps:
    z = base_logit.copy()
    z[0] = gap  # class 0 is the correct class
    hard_losses.append(hard_ce(z, 0))
    smooth_losses.append(label_smooth_ce(z, 0, epsilon=0.1))

# Hard CE minimum at gap->inf, smooth CE has finite minimum
opt_hard_gap = gaps[np.argmin(hard_losses)]
opt_smooth_gap = gaps[np.argmin(smooth_losses)]

print(f'Optimal logit gap (K=10, ε=0.1):')
print(f'  Hard CE:   gap = {opt_hard_gap:.1f} (drives to infinity)')
print(f'  Smooth CE: gap = {opt_smooth_gap:.2f} (finite optimum!)')
print(f'\nTheoretical smooth CE optimum gap: log((1-ε)K/ε) = {np.log((0.9*K)/0.1):.3f}')

check_true('Hard CE monotonically decreasing (drives logits to inf)',
           np.all(np.diff(hard_losses) <= 0.001))
check_true('Smooth CE has finite minimum', opt_smooth_gap > 0)

if HAS_MPL:
    plt.figure(figsize=(9, 4))
    plt.plot(gaps, hard_losses, color=COLORS['error'], linewidth=2, label='Hard CE (ε=0)')
    plt.plot(gaps, smooth_losses, color=COLORS['primary'], linewidth=2, label='Smooth CE (ε=0.1)')
    plt.axvline(opt_smooth_gap, color=COLORS['primary'], linestyle='--',
                linewidth=1.5, label=f'Smooth optimum = {opt_smooth_gap:.1f}')
    plt.xlabel('Logit gap (correct class - others)')
    plt.ylabel('Loss')
    plt.title('Label Smoothing Prevents Overconfident Logits')
    plt.legend(); plt.tight_layout(); plt.show()

---

## 7. KL Divergence: Forward vs Reverse, Mode-Covering vs Mode-Seeking

Forward KL ($D_{\mathrm{KL}}(p\|q)$, used in MLE) is mode-covering: $q$ must cover all modes of $p$.

Reverse KL ($D_{\mathrm{KL}}(q\|p)$, used in VI) is mode-seeking: $q$ concentrates on one mode of $p$.

In [ ]:
# === 7.1 Forward vs Reverse KL: Gaussian Approximation ===

import numpy as np

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
    COLORS = {'primary': '#0077BB', 'secondary': '#EE7733', 'tertiary': '#009988',
              'error': '#CC3311', 'neutral': '#555555', 'highlight': '#EE3377'}
except ImportError:
    HAS_MPL = False

from scipy.stats import norm

# Target p: bimodal mixture of Gaussians
def p_bimodal(x):
    return 0.5*norm.pdf(x, -2, 1) + 0.5*norm.pdf(x, 2, 1)

def kl_forward(mu_q, sig_q, n_pts=2000):
    """KL(p || q) where q = N(mu_q, sig_q^2)."""
    x = np.linspace(-6, 6, n_pts)
    dx = x[1] - x[0]
    p = p_bimodal(x)
    q = norm.pdf(x, mu_q, sig_q)
    mask = (p > 1e-10) & (q > 1e-10)
    return np.sum(p[mask] * np.log(p[mask]/q[mask])) * dx

def kl_reverse(mu_q, sig_q, n_pts=2000):
    """KL(q || p) where q = N(mu_q, sig_q^2)."""
    x = np.linspace(-6, 6, n_pts)
    dx = x[1] - x[0]
    p = p_bimodal(x)
    q = norm.pdf(x, mu_q, sig_q)
    mask = (p > 1e-10) & (q > 1e-10)
    return np.sum(q[mask] * np.log(q[mask]/p[mask])) * dx

# Grid search for best Gaussian approximation under each KL
mus = np.linspace(-4, 4, 60)
sigs = np.linspace(0.5, 4.0, 40)

fwd_kl = np.array([[kl_forward(mu, sig) for mu in mus] for sig in sigs])
rev_kl = np.array([[kl_reverse(mu, sig) for mu in mus] for sig in sigs])

# Optimal Gaussian under each KL
idx_fwd = np.unravel_index(fwd_kl.argmin(), fwd_kl.shape)
idx_rev = np.unravel_index(rev_kl.argmin(), rev_kl.shape)

mu_fwd, sig_fwd = mus[idx_fwd[1]], sigs[idx_fwd[0]]
mu_rev, sig_rev = mus[idx_rev[1]], sigs[idx_rev[0]]

print('Best Gaussian approximation to bimodal p:')
print(f'  Forward KL (mode-covering): mu={mu_fwd:.2f}, sig={sig_fwd:.2f}')
print(f'  Reverse KL (mode-seeking):  mu={mu_rev:.2f}, sig={sig_rev:.2f}')
print(f'\nExpect: Forward KL -> broad (mu≈0, sig≈3); Reverse KL -> one mode (mu≈±2, sig≈1)')

check_true('Forward KL gives broad approx (sig > 2)', sig_fwd > 2.0)
check_true('Reverse KL gives narrow approx (sig < 1.5)', sig_rev < 1.5)

if HAS_MPL:
    x_plot = np.linspace(-7, 7, 400)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    for ax, mu, sig, title, color in [
            (ax1, mu_fwd, sig_fwd, f'Forward KL (mode-covering)\nμ={mu_fwd:.2f}, σ={sig_fwd:.2f}', COLORS['primary']),
            (ax2, mu_rev, sig_rev, f'Reverse KL (mode-seeking)\nμ={mu_rev:.2f}, σ={sig_rev:.2f}', COLORS['secondary'])]:
        ax.plot(x_plot, p_bimodal(x_plot), color=COLORS['neutral'],
                linewidth=2, label='Target p(x)', linestyle='--')
        ax.plot(x_plot, norm.pdf(x_plot, mu, sig), color=color,
                linewidth=2.5, label='Best Gaussian q(x)')
        ax.set_title(title); ax.legend(); ax.set_xlabel('x')
    plt.suptitle('KL Divergence Direction Changes the Gaussian Approximation', fontsize=13)
    plt.tight_layout(); plt.show()

---

## 8. InfoNCE and Temperature: Contrastive Learning

The InfoNCE loss is a softmax cross-entropy over similarities:
$$\ell_{\text{InfoNCE}} = -\log\frac{e^{s^+/\tau}}{e^{s^+/\tau} + \sum_j e^{s_j^-/\tau}}$$

Temperature $\tau$ controls how hard the loss focuses on difficult negatives.

In [ ]:
# === 8.1 InfoNCE: Temperature Effect ===

np.random.seed(42)

def info_nce_loss(s_pos, s_neg_list, tau=0.07):
    """InfoNCE loss for one query."""
    all_scores = np.concatenate([[s_pos], s_neg_list])
    all_scores_scaled = all_scores / tau
    # Numerically stable log-softmax
    m = all_scores_scaled.max()
    log_denom = m + np.log(np.sum(np.exp(all_scores_scaled - m)))
    return -(s_pos/tau - log_denom)

def info_nce_gradient_wrt_pos(s_pos, s_neg_list, tau=0.07):
    """Gradient of InfoNCE w.r.t. s_pos. Should be p_hat_pos - 1."""
    all_scores = np.concatenate([[s_pos], s_neg_list]) / tau
    all_scores -= all_scores.max()
    p_hat = np.exp(all_scores) / np.sum(np.exp(all_scores))
    return p_hat[0] - 1.0  # gradient w.r.t. s_pos

# Scenario: 1 positive with similarity 0.8, N negatives with varying similarities
s_pos = 0.8
N_neg = 255

print('InfoNCE loss and gradients at different temperatures:')
print(f'  s_pos=0.8, N_neg={N_neg}, neg similarities ~ N(0, 0.1)')
print(f'  tau      loss      grad(s_pos)   p_hat_pos')
np.random.seed(0)
s_neg = np.random.randn(N_neg) * 0.1

for tau in [0.01, 0.07, 0.1, 0.5, 1.0]:
    loss = info_nce_loss(s_pos, s_neg, tau)
    grad = info_nce_gradient_wrt_pos(s_pos, s_neg, tau)
    all_s = np.concatenate([[s_pos], s_neg]) / tau
    all_s -= all_s.max()
    p_hat_pos = np.exp(all_s[0]) / np.sum(np.exp(all_s))
    print(f'  {tau:<8.2f} {loss:<10.4f} {grad:<14.4f} {p_hat_pos:.4f}')

# Numerical gradient verification
eps = 1e-5
tau_test = 0.07
grad_analytic = info_nce_gradient_wrt_pos(s_pos, s_neg, tau_test)
loss_plus = info_nce_loss(s_pos + eps, s_neg, tau_test)
loss_minus = info_nce_loss(s_pos - eps, s_neg, tau_test)
grad_numerical = (loss_plus - loss_minus) / (2*eps)
print(f'\nGradient verification (tau={tau_test}):')
print(f'  Analytic:  {grad_analytic:.6f}')
print(f'  Numerical: {grad_numerical:.6f}')

ok = abs(grad_analytic - grad_numerical) < 0.001
print(f"{'PASS' if ok else 'FAIL'} — InfoNCE gradient matches numerical differentiation")

---

## 9. DPO Loss: Preference Learning Without a Reward Model

The DPO loss aligns LLMs to human preferences using only the policy and a reference:
$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}\left[\log\sigma\!\left(\beta\log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)}
- \beta\log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)\right]$$

In [ ]:
# === 9.1 DPO Loss: Gradient Analysis ===

def dpo_loss(log_ratio_w, log_ratio_l, beta=0.1):
    """
    log_ratio_w: log(pi_theta(y_w|x) / pi_ref(y_w|x))
    log_ratio_l: log(pi_theta(y_l|x) / pi_ref(y_l|x))
    """
    from scipy.special import expit as sigmoid
    margin = beta * (log_ratio_w - log_ratio_l)
    return -np.log(sigmoid(margin))

def dpo_gradient_wrt_log_ratio_w(log_ratio_w, log_ratio_l, beta=0.1):
    """Gradient of DPO loss w.r.t. log_ratio_w."""
    from scipy.special import expit as sigmoid
    margin = beta * (log_ratio_w - log_ratio_l)
    p = sigmoid(margin)
    return -beta * (1 - p)  # pushes log_ratio_w up (preferred)

def dpo_gradient_wrt_log_ratio_l(log_ratio_w, log_ratio_l, beta=0.1):
    """Gradient of DPO loss w.r.t. log_ratio_l."""
    from scipy.special import expit as sigmoid
    margin = beta * (log_ratio_w - log_ratio_l)
    p = sigmoid(margin)
    return beta * (1 - p)   # pushes log_ratio_l down (dispreferred)

# Verify gradients
eps = 1e-5
lrw, lrl = 0.5, -0.3
grad_w_analytic = dpo_gradient_wrt_log_ratio_w(lrw, lrl)
grad_w_numerical = (dpo_loss(lrw+eps, lrl) - dpo_loss(lrw-eps, lrl)) / (2*eps)

print('DPO gradient verification:')
print(f'  grad w.r.t. log_ratio_w: analytic={grad_w_analytic:.6f}, numerical={grad_w_numerical:.6f}')
ok = abs(grad_w_analytic - grad_w_numerical) < 0.0001
print(f"{'PASS' if ok else 'FAIL'} — DPO gradient matches")

# Show gradient direction: increases preferred, decreases dispreferred
print(f'\nGradient directions (beta=0.1, lrw=0.5, lrl=-0.3):')
print(f'  grad(lrw) = {grad_w_analytic:.4f} < 0 (loss decreases when lrw increases — correct)')
grad_l_val = dpo_gradient_wrt_log_ratio_l(lrw, lrl)
print(f'  grad(lrl) = {grad_l_val:.4f} > 0 (loss decreases when lrl decreases — correct)')
ok2 = grad_w_analytic < 0 and grad_l_val > 0
print(f"{'PASS' if ok2 else 'FAIL'} — DPO gradients have correct signs")

# Margin analysis
margins = np.linspace(-5, 5, 300)
dpo_losses = dpo_loss(margins/2, -margins/2)

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(margins, dpo_losses, color=COLORS['primary'], linewidth=2)
    ax1.set_xlabel('Preference margin β(log_rw - log_rl)')
    ax1.set_ylabel('DPO Loss'); ax1.set_title('DPO Loss vs Preference Margin')
    ax1.grid(True, alpha=0.3)

    # Gradient as function of margin
    from scipy.special import expit as sigmoid
    grads = -0.1*(1 - sigmoid(margins))
    ax2.plot(margins, grads, color=COLORS['error'], linewidth=2)
    ax2.set_xlabel('Preference margin')
    ax2.set_ylabel('∂Loss/∂log_ratio_w')
    ax2.set_title('DPO Gradient: Largest When Margin Near 0')
    ax2.grid(True, alpha=0.3)
    plt.suptitle('DPO Loss: Direct Preference Optimisation', fontsize=13)
    plt.tight_layout(); plt.show()

---

## 10. Calibration and Proper Scoring Rules

A proper scoring rule is minimised by predicting true probabilities.
Cross-entropy and Brier score are both proper; accuracy is not.

In [ ]:
# === 10.1 Calibration: ECE and Reliability Diagrams ===

np.random.seed(42)
N = 5000

# Simulate: true positive rate is p, but model reports confidence c
# Overconfident model: c > p
true_probs = np.random.beta(1, 1, N)  # uniform true probs
y_true = (np.random.rand(N) < true_probs).astype(float)

# Model predictions: slightly overconfident (sigmoid sharpening)
def sharpen(p, T=0.5):
    """Sharpen probabilities with temperature T < 1."""
    logit = np.log(p.clip(1e-6, 1-1e-6) / (1 - p.clip(1e-6, 1-1e-6)))
    return sigmoid(logit / T)

conf_calibrated = true_probs  # perfectly calibrated
conf_overconf  = sharpen(true_probs, T=0.4)  # overconfident
conf_underconf = sharpen(true_probs, T=2.0)  # underconfident

def compute_ece(y_true, confidence, n_bins=10):
    bins = np.linspace(0, 1, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        mask = (confidence >= bins[i]) & (confidence < bins[i+1])
        if mask.sum() == 0: continue
        acc = y_true[mask].mean()
        conf = confidence[mask].mean()
        ece += mask.sum() / len(y_true) * abs(acc - conf)
    return ece

ece_cal  = compute_ece(y_true, conf_calibrated)
ece_over = compute_ece(y_true, conf_overconf)
ece_und  = compute_ece(y_true, conf_underconf)

print('Expected Calibration Error (ECE):')
print(f'  Calibrated:    ECE = {ece_cal:.4f}')
print(f'  Overconfident: ECE = {ece_over:.4f}')
print(f'  Underconfident: ECE = {ece_und:.4f}')

ok = ece_cal < ece_over and ece_cal < ece_und
print(f"{'PASS' if ok else 'FAIL'} — Calibrated model has lowest ECE")

if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, conf, title in [
            (axes[0], conf_calibrated, f'Calibrated\nECE={ece_cal:.3f}'),
            (axes[1], conf_overconf, f'Overconfident\nECE={ece_over:.3f}'),
            (axes[2], conf_underconf, f'Underconfident\nECE={ece_und:.3f}')]:
        n_bins = 10
        bins = np.linspace(0, 1, n_bins+1)
        accs = []
        confs = []
        for i in range(n_bins):
            mask = (conf >= bins[i]) & (conf < bins[i+1])
            if mask.sum() == 0:
                accs.append(0); confs.append((bins[i]+bins[i+1])/2)
            else:
                accs.append(y_true[mask].mean())
                confs.append(conf[mask].mean())
        ax.bar(confs, accs, width=0.09, alpha=0.7, color=COLORS['primary'], label='Accuracy')
        ax.plot([0,1], [0,1], 'k--', linewidth=1.5, label='Perfect')
        ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy')
        ax.set_title(title); ax.legend(fontsize=9)
    plt.suptitle('Reliability Diagrams (Calibration)', fontsize=13)
    plt.tight_layout(); plt.show()

---

## 11. Loss Landscape Geometry

We visualise the 2D loss landscape for MSE and CE and compare their curvature properties.

In [ ]:
# === 11.1 Loss Landscape Visualisation ===

np.random.seed(42)
n = 100
x_data = np.random.randn(n)
# Binary labels
y_data = (x_data + np.random.randn(n)*0.5 > 0).astype(float)

def logistic_ce_loss(w, b):
    z = w*x_data + b
    # Numerically stable
    return np.mean(np.maximum(z, 0) - y_data*z + np.log1p(np.exp(-np.abs(z))))

def logistic_mse_loss(w, b):
    z = w*x_data + b
    p = sigmoid(z)
    return np.mean((p - y_data)**2)

ws_grid = np.linspace(-3, 5, 80)
bs_grid = np.linspace(-3, 3, 80)
W, B = np.meshgrid(ws_grid, bs_grid)

CE_grid  = np.array([[logistic_ce_loss(w, b) for w in ws_grid] for b in bs_grid])
MSE_grid = np.array([[logistic_mse_loss(w, b) for w in ws_grid] for b in bs_grid])

# Find optima
ce_opt_idx  = np.unravel_index(CE_grid.argmin(),  CE_grid.shape)
mse_opt_idx = np.unravel_index(MSE_grid.argmin(), MSE_grid.shape)
w_ce, b_ce = ws_grid[ce_opt_idx[1]], bs_grid[ce_opt_idx[0]]
w_mse, b_mse = ws_grid[mse_opt_idx[1]], bs_grid[mse_opt_idx[0]]

print(f'CE  optimal: w={w_ce:.2f}, b={b_ce:.2f}')
print(f'MSE optimal: w={w_mse:.2f}, b={b_mse:.2f}')

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    for ax, grid, opt_w, opt_b, title in [
            (ax1, CE_grid, w_ce, b_ce, 'Cross-Entropy Loss Landscape'),
            (ax2, MSE_grid, w_mse, b_mse, 'MSE Loss Landscape')]:
        ct = ax.contourf(W, B, grid, levels=30, cmap='plasma')
        plt.colorbar(ct, ax=ax)
        ax.scatter([opt_w], [opt_b], color='white', s=100, zorder=5,
                   edgecolors='black', linewidths=1.5, label='Optimum')
        ax.set_xlabel('w'); ax.set_ylabel('b')
        ax.set_title(title); ax.legend()
    plt.suptitle('Loss Landscape Comparison for Logistic Regression', fontsize=13)
    plt.tight_layout(); plt.show()

---

## 12. Multi-Task Loss Balancing: Uncertainty Weighting

Kendall et al. (2018) derive optimal task weights from homoscedastic uncertainty:
$$\mathcal{L} = \sum_t \frac{1}{2\sigma_t^2}\mathcal{L}_t + \log\sigma_t$$

In [ ]:
# === 12.1 Uncertainty Weighting: Optimal Sigma Analysis ===

# Optimal sigma satisfies: d/d(sigma_t) [L_t/(2*sigma_t^2) + log(sigma_t)] = 0
# => -L_t/sigma_t^3 + 1/sigma_t = 0 => sigma_t^2 = L_t

task_losses = np.array([0.5, 1.2, 0.08])  # regression, classification, auxiliary
task_names  = ['Regression (L1=0.5)', 'Classification (L2=1.2)', 'Auxiliary (L3=0.08)']

# Grid search for optimal sigmas
sig_range = np.linspace(0.1, 3.0, 200)

def uw_total_loss(sigmas, task_losses):
    return np.sum(task_losses / (2*sigmas**2) + np.log(sigmas))

print('Optimal uncertainty weights:')
print(f'  sigma_t* = sqrt(L_t)')
print(f'  Effective weight w_t = 1/(2*sigma_t^2) = 1/(2*L_t)')
print()
sigmas_opt = np.sqrt(task_losses)
weights_opt = 1.0 / (2 * sigmas_opt**2)

for name, L, sig, w in zip(task_names, task_losses, sigmas_opt, weights_opt):
    print(f'  {name}:')
    print(f'    sigma* = sqrt({L}) = {sig:.3f}')
    print(f'    weight = 1/(2*{L}) = {w:.3f}')

# Verify: optimal sigmas minimise the uncertainty-weighted loss
# vs naive (all weights = 1)
loss_uw = uw_total_loss(sigmas_opt, task_losses)
loss_naive = np.sum(task_losses)  # w_t = 1, sigma_t = 1/sqrt(2)

print(f'\nTotal loss:')
print(f'  Uncertainty-weighted: {loss_uw:.4f}')
print(f'  Naive (all w=1):      {loss_naive:.4f}')

# Verify gradient = 0 at optimal sigma
def grad_uw(sigma, L):
    return -L / sigma**3 + 1/sigma

for L, sig in zip(task_losses, sigmas_opt):
    g = grad_uw(sig, L)
    ok = abs(g) < 1e-6
    print(f"{'PASS' if ok else 'FAIL'} — Zero gradient at sigma*={sig:.3f} for L={L}")

---

## 13. Numerical Stability: Log-Sum-Exp and Stable BCE

Numerical overflow and underflow are the most common causes of `NaN` losses.
We demonstrate the log-sum-exp trick and stable BCE implementation.

In [ ]:
# === 13.1 Log-Sum-Exp Trick: Overflow Prevention ===

def unstable_logsumexp(z):
    return np.log(np.sum(np.exp(z)))

def stable_logsumexp(z):
    m = np.max(z)
    return m + np.log(np.sum(np.exp(z - m)))

test_cases = [
    np.array([1., 2., 3.]),
    np.array([100., 200., 300.]),     # overflow
    np.array([-100., -200., -300.]),  # underflow
    np.array([1., 1000., 1.]),        # mixed
]

print('Log-sum-exp stability comparison:')
for z in test_cases:
    stable = stable_logsumexp(z)
    with np.errstate(over='ignore', invalid='ignore'):
        unstable = unstable_logsumexp(z)
    print(f'  z = {z}:')
    print(f'    stable:   {stable:.4f}')
    print(f'    unstable: {unstable}')

# Verify identity: exp(logsumexp) = sum(exp(z))
z = np.array([1., 2., 3.])
lse = stable_logsumexp(z)
ok = np.allclose(np.exp(lse), np.sum(np.exp(z)))
print(f"\n{'PASS' if ok else 'FAIL'} — exp(logsumexp(z)) == sum(exp(z))")

# Stable BCE in logit space
def stable_bce(logit, label):
    """BCE = max(z,0) - y*z + log(1+exp(-|z|))"""
    return np.maximum(logit, 0) - label*logit + np.log1p(np.exp(-np.abs(logit)))

def naive_bce(logit, label):
    p = sigmoid(logit)
    return -(label*np.log(p) + (1-label)*np.log(1-p))

print('\nBCE stability for extreme logits:')
for z_val in [-100., -10., 0., 10., 100.]:
    stable = stable_bce(z_val, 1.0)
    with np.errstate(invalid='ignore', divide='ignore'):
        naive = naive_bce(z_val, 1.0)
    print(f'  z={z_val:6.0f}: stable={stable:.4f}, naive={naive}')

ok = np.isfinite(stable_bce(100., 1.0)) and np.isfinite(stable_bce(-100., 0.))
print(f"{'PASS' if ok else 'FAIL'} — Stable BCE handles extreme logits")

---

## Summary

This notebook covered the core mathematics of loss functions:

| Section | Key Insight | Verified |
|---|---|---|
| Bayes Risk | Different losses → different optimal predictors | MSE=mean, MAE=median, Q90=90th percentile |
| Regression Losses | Gradient clipping = Huber loss; outlier robustness = 50% breakdown | Visual + fit comparison |
| NLL Framework | MSE=Gaussian NLL, BCE=Bernoulli NLL — same MLE minimiser | Grid search verification |
| CE Gradient | $\nabla_z \ell_{\text{CE}} = \hat{p} - y$ | Analytic == numerical |
| Focal Loss | $(1-\hat{p})^\gamma$ down-weights easy examples | Weight ratio analysis |
| Label Smoothing | Finite optimal logit gap prevents overconfidence | Landscape plot |
| KL Divergence | Forward=mode-covering, Reverse=mode-seeking | Gaussian approximation demo |
| InfoNCE | Temperature $\tau$ controls hard-negative focus | Gradient verification |
| DPO | Gradient increases $\pi(y_w)$, decreases $\pi(y_l)$ | Sign verification |
| Calibration | Proper scoring rules → calibrated probabilities | ECE + reliability diagrams |
| Uncertainty Weighting | $\sigma_t^* = \sqrt{\mathcal{L}_t}$ at optimum | Zero-gradient check |
| Numerical Stability | Log-sum-exp prevents NaN | Extreme logit test |

**Next**: [Exercises notebook](exercises.ipynb) — 10 graded problems on these topics.